In [1]:
import tkinter as tk
import random
import time
GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8,0)
MAX_DEPTH = 50
ANIMATION_DELAY = 450
class PuzzleIDS:

    def __init__(self):
        self.window = tk.Tk()
        self.window.title("8 Puzzle - IDS")
        self.current_board = self.generate_board()
        self.solution_path = []
        self.move_history = []
        self.animation_index = 0

        self.build_ui()
        self.update_board(self.current_board)
    def generate_children(self, state):
        empty_index = state.index(0)
        row, col = divmod(empty_index, 3)
        next_states = []
        directions = [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1)
        ]
        for dr, dc in directions:
            new_row = row + dr
            new_col = col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                swap_index = new_row * 3 + new_col
                temp = list(state)
                temp[empty_index], temp[swap_index] = (
                    temp[swap_index],
                    temp[empty_index]
                )
                next_states.append(tuple(temp))
        return next_states

    def get_direction(self, old_state, new_state):
        old_pos = old_state.index(0)
        new_pos = new_state.index(0)
        diff = new_pos - old_pos
        if diff == -3:
            return "UP"
        if diff == 3:
            return "DOWN"
        if diff == -1:
            return "LEFT"
        return "RIGHT"

    def is_solvable(self, board):
        values = [x for x in board if x != 0]
        inversions = 0
        for i in range(len(values)):
            for j in range(i + 1, len(values)):
                if values[i] > values[j]:
                    inversions += 1
        return inversions % 2 == 0

    def generate_board(self):
        while True:
            board = list(GOAL_STATE)
            random.shuffle(board)
            board = tuple(board)
            if board != GOAL_STATE and self.is_solvable(board):
                return board

    def ids(self, start_state, limit):
        stack = [(start_state, [start_state])]
        expanded_nodes = 0
        while stack:
            current_state, path = stack.pop()
            expanded_nodes += 1
            if current_state == GOAL_STATE:
                return path, expanded_nodes
            current_depth = len(path) - 1
            if current_depth < limit:
                children = self.generate_children(current_state)
                for child in children:
                    if child not in path:
                        stack.append((child, path + [child]))
        return None, expanded_nodes

    def iterative_deepening_search(self, start_state):
        total_nodes = 0
        for depth in range(MAX_DEPTH):
            result, nodes = self.ids(
                start_state,
                depth
            )
            total_nodes += nodes
            if result:
                return result, total_nodes

        return None, total_nodes
    def build_ui(self):

        container = tk.Frame(self.window)
        container.pack(padx=12, pady=12)

        left_panel = tk.Frame(container)
        left_panel.pack(side="left", padx=10)

        self.board_frame = tk.Frame(left_panel)
        self.board_frame.pack()

        self.cells = []

        for i in range(9):

            label = tk.Label(
                self.board_frame,
                width=4,
                height=2,
                font=("Arial", 24, "bold"),
                relief="solid",
                borderwidth=2
            )

            label.grid(
                row=i // 3,
                column=i % 3,
                padx=2,
                pady=2
            )

            self.cells.append(label)

        self.status_label = tk.Label(
            left_panel,
            text="Ready",
            font=("Arial", 12)
        )

        self.status_label.pack(pady=10)

        button_frame = tk.Frame(left_panel)
        button_frame.pack()

        tk.Button(
            button_frame,
            text="Shuffle",
            width=12,
            command=self.shuffle_board
        ).pack(side="left", padx=5)

        tk.Button(
            button_frame,
            text="Solve IDS",
            width=12,
            command=self.solve_puzzle
        ).pack(side="left", padx=5)

        right_panel = tk.Frame(container)
        right_panel.pack(side="right", padx=10)

        tk.Label(
            right_panel,
            text="Move List",
            font=("Arial", 14, "bold")
        ).pack()

        self.move_box = tk.Text(
            right_panel,
            width=22,
            height=20,
            font=("Consolas", 11)
        )

        self.move_box.pack()
        self.info_label = tk.Label(
            self.window,
            text="",
            font=("Arial", 11)
        )

        self.info_label.pack(pady=5)

    def update_board(self, board):
        for i, value in enumerate(board):
            self.cells[i]["text"] = "" if value == 0 else str(value)

    def shuffle_board(self):
        self.current_board = self.generate_board()
        self.solution_path = []
        self.move_history = []
        self.animation_index = 0
        self.move_box.delete("1.0", tk.END)
        self.status_label["text"] = "Ready"
        self.info_label["text"] = ""
        self.update_board(self.current_board)

    def solve_puzzle(self):
        start_time = time.perf_counter()
        path, nodes = self.iterative_deepening_search(
            self.current_board
        )
        elapsed_ms = (
            time.perf_counter() - start_time
        ) * 100
        if not path:
            return

        self.solution_path = path
        self.move_history = []

        for i in range(len(path) - 1):
            move = self.get_direction(
                path[i],
                path[i + 1]
            )

            self.move_history.append(move)
        self.animation_index = 0

        self.info_label["text"] = (
            f"Expanded Nodes: {nodes}    "
            f"Steps: {len(path) - 1}    "
            f"Time: {elapsed_ms:.2f} ms"
        )
        self.move_box.delete("1.0", tk.END)
        for step, move in enumerate(self.move_history, start=1):

            self.move_box.insert(
                tk.END,
                f"{step}. {move}\n"
            )
        self.animate_solution()

    def animate_solution(self):
        if self.animation_index >= len(self.solution_path):
            return
        board = self.solution_path[self.animation_index]
        self.update_board(board)
        if self.animation_index == 0:
            self.status_label["text"] = "Initial State"
        else:
            current_move = self.move_history[
                self.animation_index - 1
            ]
            self.status_label["text"] = (
                f"Step {self.animation_index}: "
                f"{current_move}"
            )

        self.animation_index += 1
        self.window.after(
            ANIMATION_DELAY,
            self.animate_solution
        )

    def run(self):

        self.window.mainloop()
if __name__ == "__main__":

    app = PuzzleIDS()
    app.run()